In [ ]:
import os
import requests
import json
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [ ]:
load_dotenv(override=True)

openai = OpenAI()

openai_key = os.getenv("OPENAI_API_KEY")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")

print("---- KEY CHECK ----")

if openai_key:
    print(f"✅ OPENAI_API_KEY found")
else:
    print("❌ OPENAI_API_KEY not found")

if pushover_user:
    print(f"✅ PUSHOVER_USER found")
else:
    print("❌ PUSHOVER_USER not found")

if pushover_token:
    print(f"✅ PUSHOVER_TOKEN found")
else:
    print("❌ PUSHOVER_TOKEN not found")

In [ ]:
# Load LinkedIn PDF
reader = PdfReader("linkedin.pdf")
linkedin = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

# Load summary
with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

print("Loaded data")

In [ ]:
name = "Wanjiru"

system_prompt = f"""
You are acting as {name}.

You answer questions about {name}'s career, skills, and experience.

Be professional and helpful.

If you don’t know something, say so.

If the user shows interest in contacting you, you should encourage them to share their email address so you can follow up.
If the user provides an email address, you MUST call the record_user_details tool.

## Summary:
{summary}

## LinkedIn:
{linkedin}
"""

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")

def push(message):
    requests.post(
        "https://api.pushover.net/1/messages.json",
        data={
            "user": pushover_user,
            "token": pushover_token,
            "message": message
        }
    )

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this when the user provides their email to get in touch",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string"}
        },
        "required": ["email"]
    }
}

tools = [
    {"type": "function", "function": record_user_details_json}
]

In [ ]:

def handle_tool_calls(tool_calls):
    results = []
    
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        if tool_name == "record_user_details":
            record_user_details(**arguments)
        
        results.append({
            "role": "tool",
            "content": json.dumps({"status": "ok"}),
            "tool_call_id": tool_call.id
        })
    
    return results

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] \
               + history \
               + [{"role": "user", "content": message}]
    
    done = False
    
    while not done:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )
        
        finish_reason = response.choices[0].finish_reason
        
        if finish_reason == "tool_calls":
            msg = response.choices[0].message
            tool_calls = msg.tool_calls
            
            results = handle_tool_calls(tool_calls)
            
            messages.append(msg)
            messages.extend(results)
        else:
            done = True
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat, type="messages").launch()